In [1]:
import numpy as np
import pandas as pd

In [2]:
path = r"C:\Users\JOKER\Desktop\berlin\Fächer\Summer School\Power Markets\Data"
fut = pd.read_csv(f"{path}/futures_prices.csv")
#shape = pd.read_csv(f"{path}/shape_factors.csv")
shape = pd.read_csv(f"{path}/shape_factors_v2.csv")
fut

,quote_date,delivery_year,load_type,product_type,delivery_period,delivery_start,delivery_end_exclusive,price_eur_mwh,market_activity
0,2023-09-29,2024,BASE,YEAR,CAL,2024-01-01,2025-01-01,77.26,5000
1,2023-09-29,2024,PEAK,YEAR,CAL,2024-01-01,2025-01-01,86.60,1400
2,2023-09-29,2024,BASE,QUARTER,Q1,2024-01-01,2024-04-01,66.42,3000
3,2023-09-29,2024,PEAK,QUARTER,Q1,2024-01-01,2024-04-01,77.70,840
4,2023-09-29,2024,BASE,QUARTER,Q2,2024-04-01,2024-07-01,66.23,2200
5,2023-09-29,2024,PEAK,QUARTER,Q2,2024-04-01,2024-07-01,64.76,616
6,2023-09-29,2024,BASE,QUARTER,Q3,2024-07-01,2024-10-01,74.74,1600
7,2023-09-29,2024,PEAK,QUARTER,Q3,2024-07-01,2024-10-01,69.07,448
8,2023-09-29,2024,BASE,QUARTER,Q4,2024-10-01,2025-01-01,101.39,1400
9,2023-09-29,2024,PEAK,QUARTER,Q4,2024-10-01,2025-01-01,134.42,392


In [3]:
shape

,timestamp_utc,timestamp_local,month,hour,weekday,is_peak,historical_shape_factor
0,2023-12-31T23:00:00Z,2024-01-01T00:00:00+01:00,1,0,0,0,0.701846
1,2023-12-31T23:15:00Z,2024-01-01T00:15:00+01:00,1,0,0,0,0.701846
2,2023-12-31T23:30:00Z,2024-01-01T00:30:00+01:00,1,0,0,0,0.701846
3,2023-12-31T23:45:00Z,2024-01-01T00:45:00+01:00,1,0,0,0,0.701846
4,2024-01-01T00:00:00Z,2024-01-01T01:00:00+01:00,1,1,0,0,0.678689
...,...,...,...,...,...,...,...
35131,2024-12-31T21:45:00Z,2024-12-31T22:45:00+01:00,12,22,1,0,0.938787
35132,2024-12-31T22:00:00Z,2024-12-31T23:00:00+01:00,12,23,1,0,0.822569
35133,2024-12-31T22:15:00Z,2024-12-31T23:15:00+01:00,12,23,1,0,0.822569
35134,2024-12-31T22:30:00Z,2024-12-31T23:30:00+01:00,12,23,1,0,0.822569


Take log10 of market_activity so its orders of magnitude are on a comparable scale

In [4]:
a = np.log10(fut["market_activity"].to_numpy(dtype=float))

Split the log values into two clusters with 1-D k-means; the higher cluster is eligible

In [5]:
centres = np.array([a.min(), a.max()])
for _ in range(50):
    high = np.abs(a - centres[1]) <= np.abs(a - centres[0])
    centres = np.array([a[~high].mean(), a[high].mean()])
fut["eligible"] = high
print("Activity cut-off between clusters:", 10 ** ((centres[0] + centres[1]) / 2))
print(fut[["delivery_period", "load_type", "price_eur_mwh", "market_activity", "eligible"]])

Activity cut-off between clusters: 44.87718431879305
   delivery_period load_type  price_eur_mwh  market_activity  eligible
0              CAL      BASE          77.26             5000      True
1              CAL      PEAK          86.60             1400      True
2               Q1      BASE          66.42             3000      True
3               Q1      PEAK          77.70              840      True
4               Q2      BASE          66.23             2200      True
5               Q2      PEAK          64.76              616      True
6               Q3      BASE          74.74             1600      True
7               Q3      PEAK          69.07              448      True
8               Q4      BASE         101.39             1400      True
9               Q4      PEAK         134.42              392      True
10             M01      BASE          75.32              900      True
11             M01      PEAK          88.68              252      True
12             M02      

In [6]:
granular = ["M01", "M02", "M03", "Q2", "Q3", "Q4"]
used = fut[fut["eligible"] & fut["delivery_period"].isin(granular)]

In [7]:
shape["local_date"] = shape["timestamp_local"].str[:10]
peak = shape["is_peak"].astype(bool)
shape["hpfc_eur_mwh"] = np.nan

In [8]:
used

,quote_date,delivery_year,load_type,product_type,delivery_period,delivery_start,delivery_end_exclusive,price_eur_mwh,market_activity,eligible
4,2023-09-29,2024,BASE,QUARTER,Q2,2024-04-01,2024-07-01,66.23,2200,True
5,2023-09-29,2024,PEAK,QUARTER,Q2,2024-04-01,2024-07-01,64.76,616,True
6,2023-09-29,2024,BASE,QUARTER,Q3,2024-07-01,2024-10-01,74.74,1600,True
7,2023-09-29,2024,PEAK,QUARTER,Q3,2024-07-01,2024-10-01,69.07,448,True
8,2023-09-29,2024,BASE,QUARTER,Q4,2024-10-01,2025-01-01,101.39,1400,True
9,2023-09-29,2024,PEAK,QUARTER,Q4,2024-10-01,2025-01-01,134.42,392,True
10,2023-09-29,2024,BASE,MONTH,M01,2024-01-01,2024-02-01,75.32,900,True
11,2023-09-29,2024,PEAK,MONTH,M01,2024-01-01,2024-02-01,88.68,252,True
12,2023-09-29,2024,BASE,MONTH,M02,2024-02-01,2024-03-01,60.09,800,True
13,2023-09-29,2024,PEAK,MONTH,M02,2024-02-01,2024-03-01,70.59,224,True


In [9]:
for period, block in used.groupby("delivery_period"):
    start = block["delivery_start"].iloc[0][:10]
    end = block["delivery_end_exclusive"].iloc[0][:10]
    in_period = (shape["local_date"] >= start) & (shape["local_date"] < end)
    p_base = block.loc[block["load_type"] == "BASE", "price_eur_mwh"].iloc[0]
    p_peak = block.loc[block["load_type"] == "PEAK", "price_eur_mwh"].iloc[0]

    # Count intervals and infer the off-peak price from the Base/Peak identity
    n_all = in_period.sum()
    n_peak = (in_period & peak).sum()
    n_off = n_all - n_peak
    p_off = (p_base * n_all - p_peak * n_peak) / n_off

    # Rescale the historical shape separately inside the Peak and Off-Peak slices
    peak_slice = in_period & peak
    off_slice = in_period & ~peak
    s = shape["historical_shape_factor"]
    shape.loc[peak_slice, "hpfc_eur_mwh"] = p_peak * s[peak_slice] / s[peak_slice].mean()
    shape.loc[off_slice, "hpfc_eur_mwh"] = p_off * s[off_slice] / s[off_slice].mean()
    print(period, "N_all", n_all, "N_peak", n_peak, "N_off", n_off, "implied off-peak", round(p_off, 4))

M01 N_all 2976 N_peak 1104 N_off 1872 implied off-peak 67.441
M02 N_all 2784 N_peak 1008 N_off 1776 implied off-peak 54.1305
M03 N_all 2972 N_peak 1008 N_off 1964 implied off-peak 58.6564
Q2 N_all 8736 N_peak 3120 N_off 5616 implied off-peak 67.0467
Q3 N_all 8832 N_peak 3168 N_off 5664 implied off-peak 77.9114
Q4 N_all 8836 N_peak 3168 N_off 5668 implied off-peak 82.9286


In [10]:
print("quoted vs HPFC mean:")
for period, block in used.groupby("delivery_period"):
    start = block["delivery_start"].iloc[0][:10]
    end = block["delivery_end_exclusive"].iloc[0][:10]
    in_period = (shape["local_date"] >= start) & (shape["local_date"] < end)
    q_base = block.loc[block["load_type"] == "BASE", "price_eur_mwh"].iloc[0]
    q_peak = block.loc[block["load_type"] == "PEAK", "price_eur_mwh"].iloc[0]
    print(period, "BASE", q_base, round(shape.loc[in_period, "hpfc_eur_mwh"].mean(), 4),
          "PEAK", q_peak, round(shape.loc[in_period & peak, "hpfc_eur_mwh"].mean(), 4))

quoted vs HPFC mean:
M01 BASE 75.32 75.32 PEAK 88.68 88.68
M02 BASE 60.09 60.09 PEAK 70.59 70.59
M03 BASE 63.45 63.45 PEAK 72.79 72.79
Q2 BASE 66.23 66.23 PEAK 64.76 64.76
Q3 BASE 74.74 74.74 PEAK 69.07 69.07
Q4 BASE 101.39 101.39 PEAK 134.42 134.42


In [11]:
others = fut[fut["eligible"] & ~fut["delivery_period"].isin(granular)]
print("\nquoted vs HPFC mean:")
for _, row in others.iterrows():
    in_period = (shape["local_date"] >= row["delivery_start"][:10]) & (shape["local_date"] < row["delivery_end_exclusive"][:10])
    if row["load_type"] == "PEAK":
        in_period = in_period & peak
    print(row["delivery_period"], row["load_type"], row["price_eur_mwh"],
          round(shape.loc[in_period, "hpfc_eur_mwh"].mean(), 4))


quoted vs HPFC mean:
CAL BASE 77.26 77.2594
CAL PEAK 86.6 86.6044
Q1 BASE 66.42 66.4242
Q1 PEAK 77.7 77.7018


In [12]:
shape[["timestamp_utc", "timestamp_local", "is_peak", "hpfc_eur_mwh"]].to_csv(f"{path}/hpfc.csv", index=False)

In [13]:
shape

,timestamp_utc,timestamp_local,month,hour,weekday,is_peak,historical_shape_factor,local_date,hpfc_eur_mwh
0,2023-12-31T23:00:00Z,2024-01-01T00:00:00+01:00,1,0,0,0,0.701846,2024-01-01,54.346521
1,2023-12-31T23:15:00Z,2024-01-01T00:15:00+01:00,1,0,0,0,0.701846,2024-01-01,54.346521
2,2023-12-31T23:30:00Z,2024-01-01T00:30:00+01:00,1,0,0,0,0.701846,2024-01-01,54.346521
3,2023-12-31T23:45:00Z,2024-01-01T00:45:00+01:00,1,0,0,0,0.701846,2024-01-01,54.346521
4,2024-01-01T00:00:00Z,2024-01-01T01:00:00+01:00,1,1,0,0,0.678689,2024-01-01,52.553398
...,...,...,...,...,...,...,...,...,...
35131,2024-12-31T21:45:00Z,2024-12-31T22:45:00+01:00,12,22,1,0,0.938787,2024-12-31,89.985670
35132,2024-12-31T22:00:00Z,2024-12-31T23:00:00+01:00,12,23,1,0,0.822569,2024-12-31,78.845786
35133,2024-12-31T22:15:00Z,2024-12-31T23:15:00+01:00,12,23,1,0,0.822569,2024-12-31,78.845786
35134,2024-12-31T22:30:00Z,2024-12-31T23:30:00+01:00,12,23,1,0,0.822569,2024-12-31,78.845786
